In [ ]:
!pip install -q iterative-stratification

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import log_loss

In [ ]:
SEED = 6769
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATA_DIR = '/kaggle/input/competitions/lish-moa/'
train_features = pd.read_csv(f'{DATA_DIR}train_features.csv')
scored_targets = pd.read_csv(f'{DATA_DIR}train_targets_scored.csv')
train_drug = pd.read_csv(f'{DATA_DIR}train_drug.csv')
test_features = pd.read_csv(f'{DATA_DIR}test_features.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}sample_submission.csv')

train_features["cp_time"] = train_features["cp_time"].astype(str)
test_features["cp_time"] = test_features["cp_time"].astype(str)
target_cols = [c for c in scored_targets.columns if c != "sig_id"]

In [ ]:
train_mask = train_features["cp_type"] != "ctl_vehicle"
X_train_full = train_features[train_mask].reset_index(drop=True)
y_train_full = scored_targets[train_mask].reset_index(drop=True)
drug_full = train_drug[train_mask].reset_index(drop=True)

test_mask = test_features["cp_type"] != "ctl_vehicle"
X_test_valid = test_features[test_mask].reset_index(drop=True)

In [ ]:
data = y_train_full.merge(drug_full, on="sig_id", how="left")
drug_y = data.groupby("drug_id")[target_cols].max()

fold_of_drug = {}
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(mskf.split(drug_y, drug_y)):
    for d in drug_y.index[val_idx]:
        fold_of_drug[d] = fold

data["fold"] = data["drug_id"].map(fold_of_drug)
X_train_full["fold"] = data["fold"]

In [ ]:
def build_fold_features(tr_df, va_df, te_df, var_threshold=0.95):
    """Applies QuantileTransformer, PCA, and OneHotEncoder strictly fit on training data."""
    g_cols = [c for c in tr_df.columns if c.startswith("g-")]
    c_cols = [c for c in tr_df.columns if c.startswith("c-")]
    
    g_scaler = QuantileTransformer(output_distribution="normal").fit(tr_df[g_cols])
    c_scaler = QuantileTransformer(output_distribution="normal").fit(tr_df[c_cols])
    
    g_tr, g_va, g_te = g_scaler.transform(tr_df[g_cols]), g_scaler.transform(va_df[g_cols]), g_scaler.transform(te_df[g_cols])
    c_tr, c_va, c_te = c_scaler.transform(tr_df[c_cols]), c_scaler.transform(va_df[c_cols]), c_scaler.transform(te_df[c_cols])
    
    def get_pca(X, threshold):
        cum = np.cumsum(PCA(random_state=SEED).fit(X).explained_variance_ratio_)
        n_comp = int(np.searchsorted(cum, threshold) + 1)
        pca = PCA(n_components=n_comp, random_state=SEED).fit(X)
        return pca
        
    g_pca = get_pca(g_tr, var_threshold)
    c_pca = get_pca(c_tr, var_threshold)
    
    mg_tr, mg_va, mg_te = g_pca.transform(g_tr), g_pca.transform(g_va), g_pca.transform(g_te)
    mc_tr, mc_va, mc_te = c_pca.transform(c_tr), c_pca.transform(c_va), c_pca.transform(c_te)
    
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(tr_df[["cp_dose", "cp_time"]])
    cat_tr = enc.transform(tr_df[["cp_dose", "cp_time"]])
    cat_va = enc.transform(va_df[["cp_dose", "cp_time"]])
    cat_te = enc.transform(te_df[["cp_dose", "cp_time"]])
    
    def assemble(mg, mc, cat):
        return torch.tensor(np.hstack([mg, mc, cat]), dtype=torch.float32)
        
    return assemble(mg_tr, mc_tr, cat_tr), assemble(mg_va, mc_va, cat_va), assemble(mg_te, mc_te, cat_te)

In [ ]:
class MoAMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Dropout(0.2),
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, output_dim)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
oof_preds = np.zeros((len(X_train_full), len(target_cols)))
test_preds = np.zeros((len(X_test_valid), len(target_cols)))

EPOCHS = 25
BATCH_SIZE = 128

for fold in range(5):
    print(f"--- Training Fold {fold} ---")
    
    # Split data
    tr_idx = X_train_full[X_train_full["fold"] != fold].index
    va_idx = X_train_full[X_train_full["fold"] == fold].index
    
    X_tr_df, X_va_df = X_train_full.loc[tr_idx], X_train_full.loc[va_idx]
    y_tr = torch.tensor(y_train_full.loc[tr_idx, target_cols].values, dtype=torch.float32)
    y_va = torch.tensor(y_train_full.loc[va_idx, target_cols].values, dtype=torch.float32)
    
    # Apply fold-local preprocessing
    X_tr, X_va, X_te = build_fold_features(X_tr_df, X_va_df, X_test_valid)
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_te), batch_size=BATCH_SIZE, shuffle=False)
    
    model = MoAMLP(X_tr.shape[1], len(target_cols)).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    
    # Epoch Loop
    for epoch in range(EPOCHS):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(x_batch.to(DEVICE))
            loss = criterion(preds, y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()
            
    # Validation Inference
    model.eval()
    fold_oof = []
    with torch.no_grad():
        for x_batch, _ in val_loader:
            fold_oof.append(torch.sigmoid(model(x_batch.to(DEVICE))).cpu().numpy())
    oof_preds[va_idx] = np.vstack(fold_oof)
    
    # Test Inference
    fold_test = []
    with torch.no_grad():
        for (x_batch,) in test_loader:
            fold_test.append(torch.sigmoid(model(x_batch.to(DEVICE))).cpu().numpy())
    test_preds += np.vstack(fold_test) / 5.0

In [ ]:
oof_loss = log_loss(y_train_full[target_cols].values.ravel(), oof_preds.ravel(), labels=[0,1])
print(f"\\nOverall OOF Log Loss: {oof_loss:.5f}")

In [ ]:
sub = sample_sub.copy()
sub[target_cols] = 0.0 
sub.loc[test_mask, target_cols] = test_preds
sub.to_csv("submission.csv", index=False)
print("submission.csv successfully saved.")